# Why Elo And Serve/Return Context Beat Raw Rankings

This notebook summarizes the current non-commercial baseline match-prediction results from the local `tennis.gold_match_features_asof` pipeline. It uses generated aggregate outputs only, not raw third-party data.

**Snapshot:** 2026-08-25  
**Match coverage:** 1990-01-01 to 2026-06-02  
**Primary holdout:** 2022-01-01 onward  
**Walk-forward windows:** 2016-2025 annual test windows

## Licensing And Positioning

Data is derived from Jeff Sackmann / Tennis Abstract sources via the reviewed fallback archive source key. Treat outputs as CC BY-NC-SA-bound, non-commercial research/portfolio material. These metrics are not gambling advice.

Before publishing, review `docs/model-cards/match-outcome-baseline.md` and `docs/licenses/publication-review-match-baseline.md`.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

TABLE_DIR = Path("../outputs/tables")

baseline = pd.read_csv(TABLE_DIR / "match_baseline_evaluation.csv")
walk_forward = pd.read_csv(TABLE_DIR / "match_walk_forward_evaluation.csv")
source_composition = pd.read_csv(TABLE_DIR / "match_source_composition_diagnostics.csv")
calibration = pd.read_csv(TABLE_DIR / "match_prediction_calibration.csv")

## 2022+ Holdout

The logistic baseline combines ranking, overall Elo, blended surface Elo, prior win rate, recent form, rest, and serve/return features. It is evaluated only with features known before match date. Platt and isotonic calibration are fit on validation rows only and compared on held-out test rows only.

In [ ]:
test_results = (
    baseline[baseline["split"] == "test"]
    .loc[:, ["model_name", "log_loss", "brier_score", "roc_auc", "accuracy", "sample_size"]]
    .sort_values("log_loss")
)
test_results

Caption/alt text: bar chart of 2022-onward held-out log loss by model. Lower bars are better; the logistic baseline remains the best-performing model family, while calibrated variants are shown as calibration checks rather than new primary models.

In [ ]:
px.bar(
    test_results,
    x="model_name",
    y="log_loss",
    title="2022+ Test Log Loss By Model",
    labels={"model_name": "Model", "log_loss": "Log loss"},
).update_layout(xaxis_tickangle=-35)

## Walk-Forward Stability

A single held-out period can be misleading. The walk-forward evaluation uses expanding training windows, three-year validation windows, and one-year test windows.

In [ ]:
walk_summary = (
    walk_forward.groupby("model_name", as_index=False)
    .agg(
        windows=("window_label", "nunique"),
        avg_log_loss=("log_loss", "mean"),
        avg_brier=("brier_score", "mean"),
        avg_roc_auc=("roc_auc", "mean"),
        avg_accuracy=("accuracy", "mean"),
        total_samples=("sample_size", "sum"),
    )
    .sort_values("avg_log_loss")
)
walk_summary

Caption/alt text: line chart of annual test log loss from 2016 through 2025 for the four core uncalibrated baselines. The chart checks whether the 2022-onward result is stable across rolling historical windows.

In [ ]:
core_models = ["ranking_log_difference", "overall_elo", "surface_elo", "logistic_baseline"]
px.line(
    walk_forward[walk_forward["model_name"].isin(core_models)],
    x="window_label",
    y="log_loss",
    color="model_name",
    markers=True,
    title="Annual Walk-Forward Test Log Loss",
    labels={"window_label": "Test year", "log_loss": "Log loss", "model_name": "Model"},
)

## Source-Composition Guardrail

High-performing missing-rank segments need source-composition checks before publication. Event/source flags now separate event level, qualifying/main draw, ranking population, and source coverage. Current top missing-rank segments are entirely from the fallback archive source, so treat them as diagnostic findings rather than broad model claims.

In [ ]:
missing_rank_sources = (
    source_composition[
        (source_composition["split"] == "test")
        & (source_composition["ranking_coverage"] == "missing_rank")
    ]
    .sort_values(["logistic_roc_auc", "source_sample_size"], ascending=[False, False])
    .loc[:, [
        "tour",
        "surface",
        "match_format",
        "event_level",
        "draw_stage",
        "ranking_population",
        "source_coverage",
        "source_key",
        "source_sample_size",
        "source_share",
        "serve_return_available_rate",
        "logistic_log_loss",
        "logistic_roc_auc",
    ]]
)
missing_rank_sources.head(10)

Caption/alt text: table of top missing-rank test source-composition segments. Rows show tour, surface, event level, qualifying/main-draw stage, ranking population, source coverage, fallback source share, serve/return availability, and logistic metrics. These are diagnostics, not broad model-performance claims.

## Working Takeaway

Rankings are useful but incomplete. Overall Elo gives a stronger transparent baseline, blended surface Elo handles surface adaptation without overreacting to sparse surface histories, and serve/return features help the logistic baseline when their availability is documented. The publication guardrail is to keep source-composition, event/draw, missing-rank, non-commercial, and no-betting-advice caveats attached to the article.